# Correlation Analysis — Mercari Price Prediction
> **Notebook 017** · Part of the [Mercari Price Analysis](https://github.com/) series

Explores linear relationships between engineered features and listing price. Outputs:
- Full correlation heatmap across 15 numeric features
- Ranked bar chart of per-feature correlation with price
- Multicollinearity audit (pairs with |r| > 0.8)
- Recommended feature set for modelling (Phase 3)

---
## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Feature Engineering](#2-data-loading--feature-engineering)
3. [Correlation Matrix](#3-correlation-matrix)
4. [Correlation with Price](#4-correlation-with-price)
5. [Multicollinearity Check](#5-multicollinearity-check)
6. [Key Findings](#6-key-findings)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
os.makedirs("charts", exist_ok=True)

COLORS = {
    "primary":   "#4F46E5",
    "secondary": "#0EA5E9",
    "accent":    "#F59E0B",
    "positive":  "#10B981",
    "negative":  "#EF4444",
    "neutral":   "#6B7280",
}

## 2. Data Loading & Feature Engineering

Load the Mercari sample CSV and build all numeric features used throughout this notebook.

> **Data path:** `data/mercari_sample.csv` (relative to repo root).  
> Download the raw data from [Kaggle — Mercari Price Suggestion Challenge](https://www.kaggle.com/c/mercari-price-suggestion-challenge).

In [ ]:
DATA_PATH = "data/mercari_sample.csv"   # update if your layout differs

df = pd.read_csv(DATA_PATH)

# ── Basic cleaning ─────────────────────────────────────────────────────────────
df = df.dropna(subset=["category_name"])
df["brand_name"] = df["brand_name"].fillna("No Brand")
df = df[df["price"] > 0].copy()

# ── Feature engineering ────────────────────────────────────────────────────────
df["main_category"]   = df["category_name"].str.split("/").str[0]
df["sub_category"]    = df["category_name"].str.split("/").str[1]
df["brand_name"]      = df["brand_name"].str.lower().str.strip()
df["has_description"] = df["item_description"].apply(
    lambda x: 0 if x == "No description yet" else 1
)
df["name_length"]     = df["name"].str.len()
df["name_word_count"] = df["name"].str.split().str.len()
df["desc_length"]     = df["item_description"].str.len()
df["is_branded"]      = np.where(df["brand_name"] == "no brand", 0, 1)
df["log_price"]       = np.log1p(df["price"])
df["category_depth"]  = df["category_name"].str.split("/").str.len()

condition_map = {1: "New", 2: "Like New", 3: "Good", 4: "Fair", 5: "Poor"}
df["condition_label"] = df["item_condition_id"].map(condition_map)

# ── Outlier flag (IQR method) ──────────────────────────────────────────────────
Q1, Q3 = np.percentile(df["price"], [25, 75])
IQR = Q3 - Q1
df["is_price_outlier"] = np.where(
    (df["price"] < Q1 - 1.5 * IQR) | (df["price"] > Q3 + 1.5 * IQR), 1, 0
)

# ── Category / brand aggregate price features ──────────────────────────────────
cat_avg   = df.groupby("main_category")["price"].mean()
brand_avg = df.groupby("brand_name")["price"].mean()
df["cat_avg_price"]     = df["main_category"].map(cat_avg)
df["brand_avg_price"]   = df["brand_name"].map(brand_avg)
df["price_rank_in_cat"] = (
    df.groupby("main_category")["price"]
    .rank(ascending=False, method="dense")
    .astype(int)
)
df["price_percentile"] = df["price"].rank(pct=True).round(3)

print(f"Dataset shape : {df.shape}")
print(f"Price range   : ${df['price'].min():.2f} – ${df['price'].max():.2f}")
print("Features ready for correlation analysis.")

## 3. Correlation Matrix

Compute Pearson correlations across all 15 numeric features and visualise as a heatmap.

In [ ]:
NUMERIC_FEATURES = [
    "price", "log_price", "name_length", "name_word_count",
    "desc_length", "has_description", "is_branded",
    "item_condition_id", "shipping", "is_price_outlier",
    "cat_avg_price", "brand_avg_price",
    "price_rank_in_cat", "price_percentile", "category_depth",
]

corr_matrix = df[NUMERIC_FEATURES].corr()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Full Feature Correlation Matrix", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("charts/01_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Correlation with Price

Rank features by absolute Pearson correlation with `price`. Features above the 0.1 threshold are highlighted.

In [ ]:
corr_with_price = (
    corr_matrix["price"]
    .drop("price")
    .abs()
    .sort_values(ascending=False)
    .round(3)
)

print("Feature correlation with price (absolute):")
print(corr_with_price.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

bar_colors = [
    COLORS["positive"] if v > 0.1 else COLORS["neutral"]
    for v in corr_with_price.values
]
bars = ax.barh(
    corr_with_price.index, corr_with_price.values,
    color=bar_colors, edgecolor="white"
)

for bar, val in zip(bars, corr_with_price.values):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        fontsize=9,
        fontweight="bold" if val > 0.1 else "normal",
    )

ax.axvline(0.1, color=COLORS["negative"], linestyle="--",
           linewidth=1.2, label="0.1 threshold")
ax.legend(fontsize=9)
ax.set_title("Feature Importance Preview — Correlation with Price",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Absolute Correlation with Price")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("charts/02_price_correlation_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Multicollinearity Check

Flag feature pairs whose absolute correlation exceeds **0.8** — candidates for dropping or combining before modelling.

In [ ]:
print("Feature pairs with |r| > 0.8 (review before modelling):\n")

checked = set()
flagged = []
for feat1 in NUMERIC_FEATURES:
    for feat2 in NUMERIC_FEATURES:
        if feat1 == feat2:
            continue
        pair = tuple(sorted([feat1, feat2]))
        if pair in checked:
            continue
        checked.add(pair)
        r = abs(corr_matrix.loc[feat1, feat2])
        if r > 0.8:
            flagged.append((feat1, feat2, r))
            print(f"  {feat1} ↔ {feat2}: r = {r:.3f}  ← REVIEW")

if not flagged:
    print("  None found — no multicollinearity concerns.")

# ── Recommended feature set for Phase 3 ───────────────────────────────────────
print("\nRecommended features for modelling (|r| > 0.05 with price):")
recommended = corr_with_price[corr_with_price > 0.05].index.tolist()
print(f"  Count: {len(recommended)}")
for f in recommended:
    print(f"  {f}: {corr_with_price[f]:.3f}")

## 6. Key Findings

### Summary

| # | Finding | Detail |
|---|---------|--------|
| 1 | **Strongest predictors** | `brand_avg_price`, `cat_avg_price`, and `price_percentile` dominate — aggregate price context matters more than raw listing attributes. |
| 2 | **Multicollinearity** | `price` / `log_price` / `price_percentile` / `price_rank_in_cat` are highly inter-correlated by construction — keep only **`log_price`** as the modelling target. |
| 3 | **Weak signals** | `name_length`, `name_word_count`, and `category_depth` all fall below |r| = 0.05; consider dropping them unless tree-based models are used. |
| 4 | **Shipping flag** | Slight negative correlation with price — free shipping listings tend to be cheaper or the seller absorbs cost on lower-value items. |
